In [ ]:
import numpy as np
import pandas as pd
import math
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

In [ ]:
def create_feature_clusters(origin_df, patch_features_dict, n_pca=15, n_clusters=4):
    """
    Aggregates patch features, reduces dimensionality, and clusters morphological phenotypes.
    
    Args:
        origin_df (pd.DataFrame): Must contain 'origin_id' and 'true_class'.
        patch_features_dict (dict): Mapping of origin_id to a 2D numpy array of its patch features.
    """
    origin_features = []
    patch_counts = []
    
    # 1. Aggregate features to the Origin Level (Mean Pooling)
    for orig_id in origin_df['origin_id']:
        features = patch_features_dict[orig_id]
        patch_counts.append(len(features))
        mean_feat = np.mean(features, axis=0)
        origin_features.append(mean_feat)
        
    origin_df['patch_count'] = patch_counts
    origin_features = np.array(origin_features)
    
    # 2. Dimensionality Reduction
    pca = PCA(n_components=n_pca, random_state=42)
    reduced_features = pca.fit_transform(origin_features)
    
    # 3. Morphological Clustering
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    origin_df['morph_cluster'] = kmeans.fit_predict(reduced_features)
    
    # 4. Create the composite Stratification Key
    origin_df['stratify_key'] = origin_df['true_class'].astype(str) + "_c" + origin_df['morph_cluster'].astype(str)
    
    return origin_df

In [ ]:

def greedy_patch_balanced_split(origin_df, n_folds=6):
    """
    Divides the origin photos into n_folds using an LPT greedy heuristic, ensuring:
    1. Equal distribution of origin photos per stratify_key across folds.
    2. Balanced total patch counts across all folds.
    """
    origin_df['fold'] = -1
    
    # Track the GLOBAL number of patches assigned to each fold
    global_fold_patch_counts = np.zeros(n_folds)
    
    # Process each subgroup individually to guarantee stratification
    for key, group in origin_df.groupby('stratify_key'):
        
        # LPT Heuristic: Sort largest origin photos first
        group = group.sort_values(by='patch_count', ascending=False)
        
        # Calculate the maximum number of origin photos allowed per fold for THIS specific key
        # to ensure the folds have an equal number of images from this morphological class
        n_items_in_stratum = len(group)
        max_photos_per_fold = math.ceil(n_items_in_stratum / n_folds)
        
        # Track how many photos of this specific key we've put in each fold
        local_fold_item_counts = np.zeros(n_folds)
        
        for idx, row in group.iterrows():
            # Find which folds are still allowed to take photos of this specific key
            available_folds = np.where(local_fold_item_counts < max_photos_per_fold)[0]
            
            # Edge case fallback (rare, depends on exact math.ceil dynamics)
            if len(available_folds) == 0:
                available_folds = np.arange(n_folds)
                
            # THE GREEDY CHOICE: 
            # Among the available folds, pick the one with the FEWEST total patches globally
            best_fold = available_folds[np.argmin(global_fold_patch_counts[available_folds])]
            
            # Assign and update counters
            origin_df.at[idx, 'fold'] = best_fold
            global_fold_patch_counts[best_fold] += row['patch_count']
            local_fold_item_counts[best_fold] += 1
            
    return origin_df


In [ ]:

# Simulate Data (200 origin photos)
np.random.seed(42)
simulated_origin_df = pd.DataFrame({
    'origin_id': np.arange(200),
    'true_class': np.random.choice(['dysplasia', 'without_dysplasia', 'carcinoma'], size=200)
})

# Simulate a dictionary where each origin_id maps to an array of its patch embeddings
# Images have between 1 and 50 patches, embeddings are 512D
simulated_features = {
    i: np.random.rand(np.random.randint(1, 51), 512) for i in range(200)
}

# Run Pipeline
df_clustered = create_feature_clusters(simulated_origin_df, simulated_features)
final_df = greedy_patch_balanced_split(df_clustered, n_folds=6)

# Verification Printouts
print("\n--- Origin Photos per Fold (Should be ~33) ---")
print(final_df['fold'].value_counts().sort_index())

print("\n--- Total Patches per Fold (Should be tightly balanced) ---")
print(final_df.groupby('fold')['patch_count'].sum())

print("\n--- Stratification Check for Fold 0 (Photos per subgroup) ---")
print(final_df[final_df['fold'] == 0]['stratify_key'].value_counts())